# Customer Churn Prediction (with synthetic dataset)
Steps: Create data → Load → Preprocess → Feature Engineering → Train → Test → Save model

## 0. Create synthetic dataset
This makes a fake telecom dataset with the same columns as `mlops_dataset.csv`, including some missing and invalid values. If you have the real CSV, skip this cell.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 5000

df = pd.DataFrame({
    "Customer_ID": [f"TEL{i:05d}" for i in range(1, n + 1)],
    "Gender": rng.choice(["Male", "Female"], n),
    "Senior_Citizen": rng.choice(["Yes", "No"], n, p=[0.2, 0.8]),
    "Tenure_Months": rng.integers(0, 73, n).astype(float),
    "Contract_Type": rng.choice(["Month-to-month", "One year", "Two year"], n, p=[0.55, 0.25, 0.20]),
    "Internet_Service": rng.choice(["DSL", "Fiber", "None"], n, p=[0.35, 0.45, 0.20]),
    "Payment_Method": rng.choice(["Cash", "UPI", "Credit Card", "Bank Transfer"], n),
    "Tech_Support": rng.choice(["Yes", "No"], n),
    "Online_Security": rng.choice(["Yes", "No"], n),
    "Support_Tickets_6M": rng.integers(0, 6, n).astype(float),
    "Avg_Data_Usage_GB": rng.uniform(0, 185, n).round(2),
})

# charges depend on internet type
base = df["Internet_Service"].map({"DSL": 600, "Fiber": 1000, "None": 299})
df["Monthly_Charges_INR"] = (base + rng.normal(0, 120, n)).clip(299, 1500).round(2)
df["Total_Charges_INR"] = (df["Monthly_Charges_INR"] * df["Tenure_Months"]).round(2)

# churn chance depends on the features (so models can learn something)
score = (-0.5
         + 1.2 * (df["Contract_Type"] == "Month-to-month")
         - 0.04 * df["Tenure_Months"]
         + 0.3 * df["Support_Tickets_6M"]
         + 0.6 * (df["Tech_Support"] == "No")
         + 0.4 * (df["Online_Security"] == "No")
         + 0.5 * (df["Internet_Service"] == "Fiber")
         + 0.3 * (df["Senior_Citizen"] == "Yes"))
prob = 1 / (1 + np.exp(-score))
df["Churn"] = np.where(rng.random(n) < prob, "Churn", "Not Churn")

# add messy values: blanks and invalid numbers
for c in ["Gender", "Tenure_Months", "Total_Charges_INR", "Support_Tickets_6M", "Contract_Type"]:
    df.loc[rng.choice(n, 80, replace=False), c] = np.nan
df.loc[rng.choice(n, 10, replace=False), "Monthly_Charges_INR"] = -1

df.to_csv("mlops_dataset.csv", index=False)
print("Saved mlops_dataset.csv", df.shape)
print(df["Churn"].value_counts())

## 1. Load the dataset
Reads `mlops_dataset.csv` (created above, or your own real CSV).

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("mlops_dataset.csv")
print(df.shape)
df.head()

In [ ]:
df.info()
print(df.isnull().sum())

## 2. Preprocessing (missing values, invalid values, encoding)

In [ ]:
# pandas reads the word "None" (no internet) as missing, so put it back
df["Internet_Service"] = df["Internet_Service"].fillna("None")

# remove ID column (not useful for prediction)
df = df.drop(columns=["Customer_ID"])

# make number columns really numeric (bad text becomes NaN)
num_cols = ["Tenure_Months", "Monthly_Charges_INR", "Total_Charges_INR",
            "Support_Tickets_6M", "Avg_Data_Usage_GB"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# invalid values (negative numbers) -> NaN
for c in num_cols:
    df.loc[df[c] < 0, c] = np.nan

# drop rows where target is missing, make target 0/1
df = df.dropna(subset=["Churn"])
df["Churn"] = df["Churn"].map({"Churn": 1, "Not Churn": 0})
df = df.dropna(subset=["Churn"])

# fill missing values: numbers -> median, text -> most common value
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())
cat_cols = ["Gender", "Senior_Citizen", "Contract_Type", "Internet_Service",
            "Payment_Method", "Tech_Support", "Online_Security"]
for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])

print(df.isnull().sum().sum(), "missing values left")

## 3. Feature Engineering

In [ ]:
# tenure category
df["Tenure_Group"] = pd.cut(df["Tenure_Months"], bins=[-1, 12, 24, 48, 100],
                            labels=["0-1yr", "1-2yr", "2-4yr", "4yr+"]).astype(str)

# average monthly spending
df["Avg_Monthly_Spend"] = df["Total_Charges_INR"] / df["Tenure_Months"].replace(0, 1)

# service usage indicators
df["Has_Tech_Support"] = (df["Tech_Support"] == "Yes").astype(int)
df["Has_Online_Security"] = (df["Online_Security"] == "Yes").astype(int)
df["Has_Internet"] = (df["Internet_Service"] != "None").astype(int)
df["High_Data_User"] = (df["Avg_Data_Usage_GB"] > df["Avg_Data_Usage_GB"].median()).astype(int)
df["Many_Tickets"] = (df["Support_Tickets_6M"] >= 3).astype(int)

df.head()

## 4. Categorical encoding

In [ ]:
# yes/no and gender columns -> 0/1
df["Senior_Citizen"] = df["Senior_Citizen"].map({"Yes": 1, "No": 0})
df["Gender"] = df["Gender"].map({"Male": 1, "Female": 0})
df["Tech_Support"] = df["Tech_Support"].map({"Yes": 1, "No": 0})
df["Online_Security"] = df["Online_Security"].map({"Yes": 1, "No": 0})

# other text columns -> one-hot columns
df = pd.get_dummies(df, columns=["Contract_Type", "Internet_Service",
                                 "Payment_Method", "Tenure_Group"], dtype=int)
df.head()

## 5. Train / Test split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

## 6. Train two models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "trained")

## 7. Test and compare

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

results = []
for name, model in models.items():
    pred = model.predict(X_test)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
    })
    print(name, "confusion matrix:\n", confusion_matrix(y_test, pred), "\n")

results = pd.DataFrame(results).round(3)
results

## 8. Pick best model and save it

In [ ]:
import joblib

best_name = results.sort_values("F1", ascending=False).iloc[0]["Model"]
best_model = models[best_name]
print("Best model:", best_name)

joblib.dump(best_model, "churn_model.pkl")
joblib.dump(list(X.columns), "model_columns.pkl")   # needed later for FastAPI /predict
import json, sklearn
json.dump({"best_model": best_name, "data_median": float(df["Avg_Data_Usage_GB"].median())},
          open("model_info.json", "w"))   # also needed for FastAPI
print("Saved churn_model.pkl, model_columns.pkl and model_info.json")
print("scikit-learn version used:", sklearn.__version__)

## 9. Quick prediction check

In [ ]:
sample = X_test.iloc[[0]]
pred = best_model.predict(sample)[0]
print("Prediction:", "Churn" if pred == 1 else "Not Churn")
print("Actual    :", "Churn" if y_test.iloc[0] == 1 else "Not Churn")

# Colab: download the saved files
# from google.colab import files
# files.download("churn_model.pkl")
# files.download("model_columns.pkl")
# files.download("model_info.json")